# Hazard: Hail in Europe


Auth: Timo Schmid 
Date: March 2026

This notebook will give a quick tour of the hail hazard data on the climada data API and its usage for impact modelling

## Data overview and usage

The climada data API contains hail hazard data, based high-resolution climate simulations with the COSMO model and the HAILCAST hail diagnostic, conducted within the [scClim](https://scclim.ethz.ch) research project. 
The data has been spatially perturbed to create a small probabilistic event set of 330-year duration, as described in detail in [Schmid et al. (2026)](https://doi.org/10.1016/j.cliser.2025.100630)





In [ ]:
#Import packages
import pandas as pd
import numpy as np

from climada.util.api_client import Client
from climada.entity import ImpactFuncSet, ImpactFunc



In [ ]:
#helper functions
def get_emanuel_impf(v_thresh=20, v_half=60, scale=1e-3,power=3,
                    impf_id=1, intensity=np.arange(0, 70, 1),
                    intensity_unit='mm',haz_type='HL'):
    """
    Init TC impact function using the formula of Kerry Emanuel, 2011:
    https://doi.org/10.1175/WCAS-D-11-00007.1

    Parameters
    ----------
    impf_id : int, optional
        impact function id. Default: 1
    intensity : np.array, optional
        intensity array in intensity_unit.
    v_thresh : float, optional
        first shape parameter
    v_half : float, optional
        second shape parameter
    scale : float, optional
        scale parameter, linear scaling of MDD.
        0<=scale<=1. Default: 1.0
    power : int, optional
        Exponential dependence. Default to 3 (as in Emanuel (2011))

    Raises
    ------
    ValueError

    Returns
    -------
    impf : ImpactFunc
        Impact function object
    """

    #Get the function values. Note that invalid input parameters are checked
    # within get_emanuel_vals(). (e.g. V_half <= V_thresh)
    v_temp = get_emanuel_vals(intensity,v_thresh,v_half,scale,power)

    impf = ImpactFunc(haz_type=haz_type, id=impf_id,intensity=intensity,
                        intensity_unit=intensity_unit,name='Emanuel-type')
    impf.paa = np.ones(intensity.shape)
    impf.mdd = v_temp
    return impf
def get_emanuel_vals(intensity,v_thresh=20, v_half=60, scale=1e-3,power=3):
    """Get the Emanuel-type impact function values for a given intensity array"""

    #Check whether the input parameters are valid
    if v_half <= v_thresh:
        raise ValueError('Shape parameters out of range: v_half <= v_thresh.')
    if v_thresh < 0 or v_half < 0:
        raise ValueError('Negative shape parameter.')
    if scale > 1 or scale <= 0:
        raise ValueError('Scale parameter out of range.')

    #Calculate the impact function values
    v_temp = (intensity - v_thresh) / (v_half - v_thresh)
    v_temp[v_temp < 0] = 0
    v_temp = v_temp**power / (1 + v_temp**power)
    v_temp *= scale
    return v_temp


In [ ]:
#Get an overview of the properties of the 'hail' data type on the climada API

client = Client()
data_types = client.list_data_type_infos()
dtf = pd.DataFrame(data_types)


#Select the row with data_type 'hail' and print the 'properties' column
hail_properties = dtf[dtf['data_type'] == 'hail'].iloc[0]['properties']
for row in hail_properties:
    print(row)


{'property': 'data_source', 'mandatory': True, 'description': 'radar (Radar-based daily hail hazard data), model (330-year probabilistice event set)'}
{'property': 'radar_variable', 'mandatory': False, 'description': 'MESHS (Maximum Expected Severe Hail Size), POH (Probability Of Hail)'}
{'property': 'climate_scenario', 'mandatory': False, 'description': 'REF (Reference period (2011-2021)), PGW (degree PGW scenario)'}
{'property': 'country_iso3alpha', 'mandatory': False, 'description': 'ISO3 alpha code for country'}
{'property': 'res_km', 'mandatory': False, 'description': 'spatial resolution in kilometers'}


In [ ]:
#Load the hazard

haz_ref = client.get_hazard(
    "hail",
    properties={
        "data_source": "model",
        "climate_scenario": "REF",
        "country_iso3alpha": "FRA",
    },
)

haz_fut = client.get_hazard(
    "hail",
    properties={
        "data_source": "model",
        "climate_scenario": "PGW",
        "country_iso3alpha": "FRA",
    },
)


2026-03-07 11:54:21,146 - climada.hazard.io - INFO - Reading C:\Users\timschmi\climada\data\hazard\hail\hail_FRA_REF\v2\hail_FRA_ref.h5


In [ ]:
#Plot overview of both present and future hazard

haz_ref

In [ ]:
#Set vulnerability functions as described in Schmid et al (2026); Table B.1

params_PAA = {'v_thresh': 17.8,'v_half': 39.9,'scale': 1,'power': 4.95}
params_MDD = {'a':0.064,'b':3.33e-3}

impf_setPAA = ImpactFuncSet([get_emanuel_impf(**params_PAA)])



